<a href="https://colab.research.google.com/github/KayraOrbay/GROUP37_CENG313/blob/eda-guncelleme/assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Colab çalışıyor")

Colab çalışıyor


In [ ]:
%pip install -q uv

In [ ]:
!git clone https://github.com/KayraOrbay/GROUP37_CENG313.git

Cloning into 'GROUP37_CENG313'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 17 (delta 3), reused 7 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), done.
Resolving deltas: 100% (3/3), done.


In [ ]:
%cd /content/GROUP37_CENG313
!ls

/content/GROUP37_CENG313
assignment1_eda_baseline.ipynb	README.md  requirements.txt  src


In [ ]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
assert requirements.is_file(), "Repo klasöründe requirements.txt bulunamadı."

subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--system",
        "--python", sys.executable,
        "-r", str(requirements),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--system', '--python', '/usr/bin/python3', '-r', 'requirements.txt'], returncode=0)

In [ ]:
# Proje yapisi ve kurulum kontrolu
from pathlib import Path
import sys

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'src klasoru bulunamadi.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src
print('src import basarili')
print('src dosyalari:', [str(p.relative_to(repo_root)) for p in (repo_root / 'src').rglob('*') if p.is_file() and '__pycache__' not in str(p)])
for name in ['README.md', 'requirements.txt', '.gitignore']:
    p = repo_root / name
    print('\n---', name, '---')
    print(p.read_text()[:12000] if p.exists() else 'Dosya yok')

src import basarili
src dosyalari: ['src/data.py', 'src/models.py', 'src/_init_.py', 'src/evaluation.py']

--- README.md ---
# image-matching-challenge-2025
"https://www.kaggle.com/competitions/image-matching-challenge-2025"

Ayşegül Pınar Yılmaz 
23118080004
Cherrylin

Kayra Orbay
23118080017 
KayraOrbay


--- requirements.txt ---


--- .gitignore ---



In [ ]:
# Gercek proje modullerini kontrol et
import importlib
for module_name in ['src.data', 'src.models', 'src.evaluation']:
    module = importlib.import_module(module_name)
    print(module_name, 'OK:', module.__file__)
    print(Path(module.__file__).read_text()[:8000] or '(Dosya bos)')

# Veri, model agirliklari ve erisim bilgilerini Git takibinden haric tut
ignore_path = repo_root / '.gitignore'
existing = ignore_path.read_text() if ignore_path.exists() else ''
rules = ['data/', 'datasets/', 'weights/', 'checkpoints/', '*.pt', '*.pth', '*.ckpt', '*.h5', '*.onnx', '*.safetensors', '*.zip', '.env', '.env.*', '!.env.example', 'secrets/', 'kaggle.json', 'credentials.json', 'service-account*.json', '*.pem', '*.key', '__pycache__/', '*.pyc', '.ipynb_checkpoints/']
missing = [rule for rule in rules if rule not in existing.splitlines()]
if missing:
    ignore_path.write_text(existing + ('\n' if existing and not existing.endswith('\n') else '') + '\n'.join(missing) + '\n')
print('\n.gitignore hazir:', ignore_path)
print('Veri klasorleri:', [str(p) for p in [repo_root / 'data', Path('/content/data'), Path('/content/drive/MyDrive')] if p.exists()])

src.data OK: /content/GROUP37_CENG313/src/data.py
(Dosya bos)
src.models OK: /content/GROUP37_CENG313/src/models.py
(Dosya bos)
src.evaluation OK: /content/GROUP37_CENG313/src/evaluation.py
(Dosya bos)

.gitignore hazir: /content/GROUP37_CENG313/.gitignore
Veri klasorleri: []


In [ ]:
# Kaggle train_labels.csv dosyasini oku; ek paket gerektirmez.
from pathlib import Path
import importlib
import sys
from google.colab import files

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'Once repo klonlama ve kurulum hucrelerini calistirin.'
loader_source = '''import csv
from pathlib import Path

def read_train_labels(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError("Beklenen train_labels.csv sutunlari eksik.")
        rows = list(reader)
    if not rows:
        raise ValueError("Etiket dosyasi bos.")
    return rows
'''
module_path = repo_root / 'src' / 'data.py'
existing = module_path.read_text()
assert not existing.strip() or existing == loader_source, 'data.py degismis; uzerine yazilmadi.'
module_path.write_text(loader_source)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src.data
importlib.reload(src.data)

data_dir = repo_root / 'data'
data_dir.mkdir(exist_ok=True)
labels_path = data_dir / 'train_labels.csv'
if not labels_path.exists():
    print('Kaggle sitesinden indirilen train_labels.csv dosyasini secin.')
    uploaded = files.upload()
    if 'train_labels.csv' not in uploaded:
        raise ValueError('train_labels.csv secilmedi.')
    labels_path.write_bytes(uploaded['train_labels.csv'])

rows = src.data.read_train_labels(labels_path)
print('Veri okuma basarili. Satir sayisi:', len(rows))
print('Sutunlar:', list(rows[0]))
print('Ilk kayit:', rows[0])
print('Veri kumesi sayisi:', len({row['dataset'] for row in rows}))

Kaggle sitesinden indirilen train_labels.csv dosyasini secin.


Saving train_labels.csv to train_labels.csv
Veri okuma basarili. Satir sayisi: 1945
Sutunlar: ['dataset', 'scene', 'image', 'rotation_matrix', 'translation_vector']
Ilk kayit: {'dataset': 'imc2023_haiper', 'scene': 'fountain', 'image': 'fountain_image_116.png', 'rotation_matrix': '0.122655949;0.947713775;-0.294608417;0.122670643;0.280096125;0.952101924;0.984838784;-0.152920769;-0.081901202', 'translation_vector': '0.093771314;-0.803560988;2.062001533'}
Veri kumesi sayisi: 13
